# 🗂️ Notebook 2 — Data Model, APIs & Why Artifacts Must Be Immutable

In Notebook 1 we built a pipeline runner. Now we give the system a **memory** and an **API**.

You'll learn:

1. The core entities: **Pipeline, Run, Stage, Artifact, Deployment**, and how they relate.
2. A tiny typed **HTTP-style API** (pure Python — no web server needed).
3. Why **`:latest` is a lie** and what a content-addressed artifact actually looks like.
4. How rollback works when artifacts are immutable (almost free!).

## 🛠️ Setup

```bash
cd 06-system-designs/code-deployment
uv sync
```

Pick the `.venv` kernel in VS Code. Reload the window if it doesn't appear.

## 1. Entities at a glance

```
Pipeline  (config, versioned with the repo)
  └── Run        (one execution of the pipeline for one commit)
        └── Stage   (one step of that run: build / test / deploy …)
              └── (optionally) produces Artifact(s)
                                         │
                                         ▼
                                   Deployment   (artifact pinned to an env)
```

- A **Pipeline** is a *recipe*. It lives in the repo (e.g. `.github/workflows/ci.yml`).
- A **Run** is what happens when someone triggers that recipe on a specific commit.
- An **Artifact** is the *output* of a run (a container image, a zip, a binary). It should be **immutable** and **content-addressed** (its name is a hash of its contents).
- A **Deployment** says "artifact `X` is installed in environment `Y` with strategy `Z`".

## 2. Pydantic models — typed, validated, serialisable

Pydantic catches bad data *at the edge* of the system. If a webhook sends a malformed payload, Pydantic raises before we corrupt the DB.

In [ ]:
from __future__ import annotations
from datetime import datetime, timezone
from typing import Literal, Optional
from pydantic import BaseModel, Field, ValidationError, field_validator

Status   = Literal["queued", "running", "passed", "failed", "cancelled"]
Env      = Literal["dev", "staging", "prod"]
Strategy = Literal["rolling", "canary", "bluegreen"]

class StageDef(BaseModel):
    name: str
    cmd: str                              # e.g. "make test"
    depends_on: list[str] = []

class Pipeline(BaseModel):
    repo: str                             # "acme/payments"
    stages: list[StageDef]

    @field_validator("stages")
    @classmethod
    def _unique_stage_names(cls, v):
        names = [s.name for s in v]
        if len(names) != len(set(names)):
            raise ValueError("stage names must be unique")
        return v

class Artifact(BaseModel):
    repo: str
    sha256: str = Field(min_length=64, max_length=64)   # content hash
    size_bytes: int
    created_at: datetime

    @property
    def ref(self) -> str:
        # Human-readable, still immutable: "acme/payments@sha256:abc…"
        return f"{self.repo}@sha256:{self.sha256[:12]}"

class Run(BaseModel):
    id: int
    pipeline: str                         # repo name for simplicity
    commit_sha: str
    status: Status = "queued"
    started_at: Optional[datetime] = None
    finished_at: Optional[datetime] = None
    artifact: Optional[Artifact] = None

class Deployment(BaseModel):
    id: int
    env: Env
    artifact_ref: str                     # Artifact.ref — never "latest"
    strategy: Strategy = "rolling"
    at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

# Smoke test: Pydantic must reject duplicate stage names.
try:
    Pipeline(repo="a/b", stages=[StageDef(name="x", cmd="true"),
                                 StageDef(name="x", cmd="true")])
except ValidationError as e:
    # e.errors() is the structured form — much nicer than scraping str(e).
    print("✅ rejected as expected:", e.errors()[0]["msg"])

**Why the extra validation?**
In CI/CD you usually import JSON written by a human. If two stages share a name, your dependency graph is ambiguous. Catching it *at parse time* beats debugging a weird deploy 30 min later.

## 3. HTTP-style API (no web server needed)

Below is the shape of endpoints a real service would expose. We implement them as plain Python functions so we can call them straight from the notebook.

| Method | Path | Purpose |
|---|---|---|
| `POST` | `/hooks/push` | Git webhook — creates a `Run` |
| `GET`  | `/runs/{id}` | Run status |
| `POST` | `/artifacts` | Register a built artifact |
| `POST` | `/deployments` | Deploy a specific artifact to an env |
| `POST` | `/deployments/{id}/rollback` | Roll back to the previous deployment in that env |
| `GET`  | `/deployments?env=prod` | Current deployment per env |

In [ ]:
# A toy in-memory service (stand-in for FastAPI + Postgres)
import hashlib, itertools

class DeployService:
    def __init__(self):
        self._runs: dict[int, Run] = {}
        self._artifacts: dict[str, Artifact] = {}        # ref -> Artifact
        self._deployments: dict[int, Deployment] = {}
        self._by_env: dict[str, list[Deployment]] = {e: [] for e in ("dev", "staging", "prod")}
        self._run_id = itertools.count(1)
        self._dep_id = itertools.count(1)

    # POST /hooks/push
    def on_push(self, repo: str, commit_sha: str) -> Run:
        run = Run(id=next(self._run_id), pipeline=repo, commit_sha=commit_sha,
                  status="queued", started_at=datetime.now(timezone.utc))
        self._runs[run.id] = run
        return run

    # POST /artifacts
    def register_artifact(self, repo: str, content: bytes) -> Artifact:
        sha = hashlib.sha256(content).hexdigest()
        art = Artifact(repo=repo, sha256=sha, size_bytes=len(content),
                       created_at=datetime.now(timezone.utc))
        # Content-addressed: if two builds produce identical bytes, we dedupe for free.
        self._artifacts.setdefault(art.ref, art)
        return self._artifacts[art.ref]

    # POST /deployments
    def deploy(self, env: Env, artifact_ref: str, strategy: Strategy = "rolling") -> Deployment:
        if artifact_ref not in self._artifacts:
            raise ValueError(f"unknown artifact: {artifact_ref}")
        dep = Deployment(id=next(self._dep_id), env=env,
                         artifact_ref=artifact_ref, strategy=strategy)
        self._deployments[dep.id] = dep
        self._by_env[env].append(dep)
        return dep

    # POST /deployments/{id}/rollback
    def rollback(self, env: Env) -> Deployment:
        history = self._by_env[env]
        if len(history) < 2:
            raise RuntimeError("nothing to roll back to")
        previous = history[-2]                         # N-1
        # Rollback = redeploy the previous artifact. Because artifacts are immutable,
        # this is *exactly* the bits we ran before — no surprises.
        return self.deploy(env, previous.artifact_ref, strategy="rolling")

    # GET /deployments?env=prod
    def current(self, env: Env) -> Optional[Deployment]:
        return self._by_env[env][-1] if self._by_env[env] else None

svc = DeployService()
print("service ready")

## 4. Demo: build two versions, deploy, roll back

Notice we **never mutate** an artifact once registered. The "v2" bug is fixed by deploying a *different* artifact — not by patching v1 in place.

In [ ]:
# Build v1 and v2 as toy "artifacts" (just bytes)
run1 = svc.on_push("acme/payments", commit_sha="aaaa111")
art_v1 = svc.register_artifact("acme/payments", b"payments-binary-v1")
print("v1 ref:", art_v1.ref)

run2 = svc.on_push("acme/payments", commit_sha="bbbb222")
art_v2 = svc.register_artifact("acme/payments", b"payments-binary-v2 (oh no, a bug)")
print("v2 ref:", art_v2.ref)

# Deploy v1, then v2
svc.deploy("prod", art_v1.ref, strategy="rolling")
svc.deploy("prod", art_v2.ref, strategy="canary")
print("current prod:", svc.current("prod").artifact_ref)

# v2 misbehaves — roll back
rolled = svc.rollback("prod")
print("after rollback, prod:", svc.current("prod").artifact_ref)
assert svc.current("prod").artifact_ref == art_v1.ref
print("✅ back to v1")

## 5. Why `:latest` is a lie (and what to use instead)

### 🔴 BAD — mutable tag

```yaml
image: acme/payments:latest   # what does this mean *tomorrow*?
```

`:latest` is just a pointer **anyone** can move. Consequences:

- The image that ran in staging can be **different** from the one now running in prod, even though both say `:latest`.
- Rollback is unreliable — you can't go back to "yesterday's `:latest`" because it's been overwritten.
- Two pods started 30 seconds apart during a rollout can end up with **different versions** of your code. 🙀

### 🟢 GOOD — content-addressed digest

```yaml
image: acme/payments@sha256:8c7e34…   # bytes-identical, forever
```

- The digest is a hash of the image's contents. If the content changes, the hash changes.
- Two servers pulling the same digest are *guaranteed* to get the same bits.
- Rollback = redeploy the previous digest. No ambiguity.

In [ ]:
# Prove it: hashing the same bytes gives the same ref; different bytes give different refs.
a = svc.register_artifact("acme/payments", b"payments-binary-v1")   # already seen
b = svc.register_artifact("acme/payments", b"payments-binary-v1")   # identical bytes
c = svc.register_artifact("acme/payments", b"payments-binary-v1 ")  # trailing space!

print("a == b ?", a.ref == b.ref)   # True  → deduped
print("a == c ?", a.ref == c.ref)   # False → different bytes, different hash
print(a.ref)
print(b.ref)
print(c.ref)

## 6. API edge cases worth testing

Real CI systems invest a lot in these; the notebook just surfaces them so you know they exist.

- **Idempotent webhooks.** Git can resend a push; the `on_push` handler must be safe to call twice (key on `(repo, commit_sha)`).
- **Concurrency.** Two runs may try to register the same artifact at once — our `setdefault` call handles that.
- **Authorization.** Only members of a team may `POST /deployments` for that team's service.
- **Audit log.** Every deployment & rollback is persisted with *who*, *what*, *when*, *why*.
- **Secrets.** They never appear in the pipeline config; fetch from a secrets manager at run time and mask in logs.

## 🔑 Key takeaways

1. Model your system as **typed entities** (Pydantic / dataclasses / protobuf) — not dicts.
2. **Artifacts are immutable and content-addressed.** Deploy by digest, never by tag.
3. **Rollback is just redeploying a previous immutable artifact.** That's why immutability buys you safety.
4. Keep the API small and resource-oriented: `runs`, `artifacts`, `deployments`, `rollbacks`.

Up next: *how* do we actually roll forward and back safely? **Rolling, blue/green, canary & SLO-gated auto-rollback →**